# Using MLflow AI Gateway with AutoGen

[MLflow AI Gateway](https://mlflow.org/docs/latest/llms/gateway/index.html) is a database-backed LLM proxy built into the MLflow tracking server (MLflow ≥ 3.0). It gives you a **single OpenAI-compatible endpoint** that can route to dozens of LLM providers — OpenAI, Anthropic, Gemini, Mistral, Bedrock, Ollama, and more.

Key features:
- **Multi-provider routing** — switch models without changing agent code
- **Secrets management** — provider API keys stored encrypted on the server; your application sends no provider keys
- **Fallback & retry** — automatic failover to backup models
- **Budget tracking** — per-endpoint or per-user token budgets
- **Usage tracing** — every call logged as an MLflow trace automatically

Because MLflow Gateway speaks the OpenAI API, you can use `OpenAIChatCompletionClient` with a custom `base_url` to point any AutoGen agent at it.

## Prerequisites

1. **Start an MLflow server:**
   ```bash
   pip install mlflow
   mlflow server --host 127.0.0.1 --port 5000
   ```

2. **Create a gateway endpoint** via the MLflow UI at [http://localhost:5000](http://localhost:5000):  
   Navigate to **AI Gateway → Create Endpoint**, give it a name (e.g. `my-chat-endpoint`), select a provider and model, and enter your API key (stored encrypted on the server).

   ![Create Endpoint UI](mlflow_gateway_images/create_endpoint.png)

   See the [MLflow AI Gateway documentation](https://mlflow.org/docs/latest/genai/governance/ai-gateway/) for advanced setup options including programmatic endpoint creation via the REST API.

## Installation

In [ ]:
pip install -U 'autogen-agentchat' 'autogen-ext[openai]'

## Connect to MLflow Gateway

Use `OpenAIChatCompletionClient` with:
- `base_url` pointing to the MLflow Gateway OpenAI-compatible endpoint
- `model` set to your **gateway endpoint name**
- `api_key` set to any non-empty string (the gateway manages provider keys server-side)

In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

MLFLOW_GATEWAY_URL = "http://localhost:5000"
ENDPOINT_NAME = "my-chat-endpoint"  # the endpoint name you created in MLflow

model_client = OpenAIChatCompletionClient(
    model=ENDPOINT_NAME,
    base_url=f"{MLFLOW_GATEWAY_URL}/gateway/openai/v1",
    api_key="unused",  # provider keys are stored on the MLflow server
    model_capabilities={
        "json_output": False,
        "vision": False,
        "function_calling": True,
    },
)

## Single-turn Chat Example

Use the model client directly to verify the connection:

In [ ]:
from autogen_core.models import UserMessage

result = await model_client.create(
    messages=[UserMessage(content="What is MLflow AI Gateway?", source="user")]
)
print(result.content)

## Multi-Agent Chat Example

Here we create two agents — a user proxy and an assistant — and run a short conversation through MLflow Gateway.

In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import MaxMessageTermination

# Create the assistant using the MLflow Gateway client
assistant = AssistantAgent(
    name="assistant",
    model_client=model_client,
    system_message="You are a helpful AI assistant. Keep answers concise.",
)

# Run a quick conversation
termination = MaxMessageTermination(max_messages=3)
team = RoundRobinGroupChat([assistant], termination_condition=termination)

await Console(team.run_stream(task="Explain LLM gateways in two sentences."))
await model_client.close()

## Streaming

MLflow Gateway supports streaming. AutoGen uses streaming automatically when available.

In [ ]:
from autogen_core.models import UserMessage

async for chunk in model_client.create_stream(
    messages=[UserMessage(content="Write a haiku about LLM gateways.", source="user")]
):
    if hasattr(chunk, 'content') and chunk.content:
        print(chunk.content, end="", flush=True)

## Gateway Features

All of these are configured in the MLflow UI — no code changes needed in your AutoGen application:

| Feature | Description |
|---------|-------------|
| **Fallback** | If the primary model fails or is rate-limited, the gateway retries with a backup model automatically |
| **Traffic splitting** | Route X% of requests to model A and Y% to model B for A/B testing |
| **Budget tracking** | Set token/cost limits per endpoint or per user |
| **Usage tracing** | Every call is logged as an MLflow trace — inputs, outputs, latency, token counts |

Your `model=ENDPOINT_NAME` value stays the same regardless of which provider or model the gateway routes to behind the scenes.

### Budget Tracking

![Budget Tracking UI](mlflow_gateway_images/budget_tracking.png)

### Usage Tracing

The **Usage dashboard** shows request volume, latency, and error rates at a glance:

![Usage Dashboard](mlflow_gateway_images/usage_dashboard.png)

The **Logs tab** lists every traced request with its response, token counts, execution time, and status:

![Usage Traces](mlflow_gateway_images/usage_traces.png)

Click any trace to see the full **request and response detail**:

![Trace Detail](mlflow_gateway_images/usage_trace_detail.png)